# Chapter 21 — Excel with pandas: reading, writing, sheets, and import policies

**Book role:** Excel workflows  
**Suggested book space:** about 10 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Read Excel workbooks and selected sheets into DataFrames.
- Handle headers, dtypes, formulas, merged-cell artifacts, and date parsing.
- Write multiple output sheets and explain the overwrite semantics.
- Build an Excel import/export workflow with validation and feedback.

## 1. Excel is a workbook container, not simply a CSV with formatting

An Excel workbook can contain multiple sheets, formulas, number formats, merged cells, charts, tables, filters, and hidden data. When pandas imports a sheet, the goal is to produce a rectangular DataFrame, not recreate the visual workbook. Formulas may be read as cached values depending on the engine and file contents; Excel formatting does not guarantee that numeric-looking cells are actually stored as numbers.

Before import, choose the sheet by name or index, decide whether the first row is a header, identify key columns that must remain strings (for example, postal codes or IDs), and set date/decimal policies deliberately. Inspect `shape`, `columns`, `dtypes`, and key fields after loading. For complex workbooks, use openpyxl for structural inspection before pandas normalization.

### Worked example 1: Read a workbook and inspect sheet names

The guard makes this example safe when the course pack does not include a workbook. Students can point it at their own file and record the sheet-selection decision.

In [ ]:
from pathlib import Path
import pandas as pd

workbook_path = Path("data") / "sales_workbook.xlsx"
if workbook_path.exists():
    book = pd.ExcelFile(workbook_path)
    print("Sheets:", book.sheet_names)
    sheet_name = book.sheet_names[0]
    frame = pd.read_excel(book, sheet_name=sheet_name)
    print("Selected sheet:", sheet_name)
    print("Shape:", frame.shape)
    print(frame.dtypes)
else:
    print("Place a workbook at", workbook_path.resolve(), "or choose one using a file dialog.")

## 2. Reading multiple sheets and preserving provenance

`pd.read_excel()` reads one sheet by default and supports a sheet name or `sheet_name=None` to load all sheets into a dictionary of DataFrames. A workbook with different grains on different sheets should not be concatenated casually. For example, `Orders`, `Products`, and `Summary` are different entities and require separate handling.

Record the workbook path, selected sheet, header row, row count, source columns, import timestamp, and validation outcome. When importing formulas, decide whether the user expects formulas or displayed values. `pandas.read_excel` is designed for data extraction; the workbook editor in openpyxl can inspect formulas and workbook features, subject to the file’s content and calculation state.

### Worked example 2: Write summary and detail sheets

This creates a new workbook with two sheets. It does not preserve formatting or unrelated sheets from an existing workbook. Install openpyxl for `.xlsx` support and test outputs under a clear overwrite policy.

In [ ]:
from pathlib import Path
import pandas as pd

sales = pd.DataFrame({"region": ["North", "South", "North"], "revenue": [100.0, 150.0, 90.0]})
summary = sales.groupby("region", as_index=False).agg(revenue=("revenue", "sum"))
output_path = Path("output") / "sales_report.xlsx"
output_path.parent.mkdir(exist_ok=True)
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    sales.to_excel(writer, sheet_name="Filtered Data", index=False)
print("Created:", output_path.resolve())

## 3. Writing is an operation with risks

`DataFrame.to_excel()` can create a workbook or append/replace sheets when paired with an `ExcelWriter` engine and options. Define whether the report should overwrite an existing file, add a new sheet, or create a new version. An analytics export often benefits from separate sheets for `Summary`, `Filtered Data`, and `Data Quality`, with consistent labels and a data dictionary.

Do not tell the user that an export succeeded until the writer has closed successfully. For sensitive data, confirm that hidden or unrelated sheets are not unintentionally carried into a new report. A newly written workbook need not copy every feature of the original. If layout, formulas, styles, or charts must be preserved, use openpyxl or a dedicated template workflow and test the output in Excel-compatible software.

### Worked example 3: Choose explicit dtype for identifier columns

The exact dtype argument depends on the source schema. Do not indiscriminately convert every numeric column to a string; measures should remain numeric for calculations.

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("data") / "id_example.csv"
# Excel ingestion example: force an identifier to text if that matches the business schema.
# frame = pd.read_excel("source.xlsx", dtype={"postal_code": "string", "customer_id": "string"})
# Use parse_dates=["order_date"] only after confirming the column's date semantics.
print("Keep IDs as strings when leading zeros or non-arithmetic meaning must be preserved.")

## Guided lab: Lab: controlled Excel import and export

Build a file-dialog-based Excel import screen. Let the user select a workbook and a sheet, preview the first 20 rows, choose identifier columns that must remain text, and validate the schema. Export `Summary`, `Filtered Data`, `Data Quality`, and `Data Dictionary` sheets. Add overwrite handling, clear cancellation behavior, and a post-export check of sheet names and row counts.

## Review questions and practice

1. Why does pandas not recreate Excel’s full visual layout?
2. When would `sheet_name=None` be useful?
3. Why should IDs sometimes be imported as strings?
4. What should the UI do if a user cancels a file dialog?
5. What is the difference between writing a report and modifying a template workbook?

## Further reading

- pandas Excel I/O: https://pandas.pydata.org/docs/user_guide/io.html#excel-files
- pandas read_excel: https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html
- pandas ExcelWriter: https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.